In [1]:
import os
from collections import defaultdict
from itertools import chain

import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
    SingularityVote,
)
from social_groups.reporting.parsing import (
    AnswerOptions,
    AnswerParser,
    AnswerComparer,
)
from social_groups.directories import REPORTING_DIR
from social_groups.reporting.plots.config import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme

from social_groups.reporting.heterogenous_comparison.data_retrieval import (
    apply_parsing_and_group_decision
)

%load_ext autoreload
%autoreload 2

In [2]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "wrong"

In [3]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling=triple_underscore_handling
)

## Analzying Basic Group Behaviour

In [2]:
from social_groups.analysis.definitions import defs

mad_frame = defs.load_fn().load_asset_value("hetero_mad")
baseline_frame = defs.load_fn().load_asset_value("baseline")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-02-18 11:48:58 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/hetero_mad.parquet using PolarsParquetIOManager...


In [3]:
mad_frame.head()

id,run_id,question_id,phoenix_span_id,phoenix_span_url,run_identifier,answers_at_beginning,answers_at_end,experiment_id,experiment_configuration_json,meta_info_json,name,original_question_id,category,question,answer_string,model_names
i64,i64,i64,str,str,str,list[str],list[str],i64,str,str,str,i64,str,str,str,list[str]


In [6]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(
    mad_frame, parser, comparer, group_reply
)

table_page_46 = pl.concat(
    [
        (
            mad_analysis.group_by("group_constellation")
            .agg(accuracy=pl.col("is_correct").mean())
            .with_columns(origin=pl.lit("(MAD)"))
        ),
        (
            baseline_frame.with_columns(
                group_constellation=pl.col("model_name").replace(
                    MODEL_NAME_TO_LETTER_MAPPING
                ),
                origin=pl.lit("(baseline)"),
            ).drop("model_name")
        ),
    ],
    how="diagonal",
)

table_page_46.write_csv(output_dir / "llm_based_table_page_46.csv")

table_page_46

Number of unparsable answers:
shape: (3, 3)
┌─────────────────┬─────────┬─────────────────┐
│ model_name      ┆ no_null ┆ null_percentage │
│ ---             ┆ ---     ┆ ---             │
│ str             ┆ u32     ┆ f64             │
╞═════════════════╪═════════╪═════════════════╡
│ Qwen/Qwen3-0.6B ┆ 90      ┆ 10.0            │
│ Qwen/Qwen3-4B   ┆ 88      ┆ 12.0            │
│ Qwen/Qwen3-14B  ┆ 93      ┆ 7.0             │
└─────────────────┴─────────┴─────────────────┘


model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.35
"""Qwen/Qwen3-4B""",0.59
"""Qwen/Qwen3-14B""",0.64


In [ ]:
original_table_page46 = pl.DataFrame(
    {
        "group_constellation": [
            "HHH",
            "HHM",
            "HHL",
            "HML",
            "HMM",
            "H",
            "HLL",
            "MMM",
            "M",
            "MML",
            "MLL",
            "L",
            "LLL",
        ],
        "score (-115 to 115)": [80, 74, 67, 64, 61, 60, 56, 48, 42, 39, 37, 25, 21],
    }
)
original_table_page46.write_csv(output_dir / "human_based_table_page_46.csv")
original_table_page46

In [7]:
decision_schemes = (
    mad_analysis.group_by("group_constellation")
    .map_groups(
        lambda g: calculate_decision_scheme(
            g,
            AnalysisColumn.parsed_individual_answers_before.value,
            AnalysisColumn.parsed_individual_answers_after.value,
            "answer_string",
            group_reply,
            comparer,
        ).select(
            pl.lit(g["group_constellation"].unique().item()).alias(
                "group_constellation"
            ),
            "Correct Members Beginning",
            "correct",
            "incorrect",
        )
    )
    .sort("group_constellation")
)

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

shape: (3_400, 17)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ category   ┆ question   ┆ answer_str ┆ model_name │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ ---        ┆ ---        ┆ ing        ┆ s          │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ str        ┆ str        ┆ ---        ┆ ---        │
│      ┆        ┆             ┆ str        ┆   ┆            ┆            ┆ str        ┆ list[str]  │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 246  ┆ 3      ┆ 11          ┆ bf94cefe4f ┆ … ┆ philosophy ┆ Q: The     ┆ C          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 7d6a67     ┆   ┆            ┆ theory     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ that says  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ mental…    ┆            ┆            │
│ 247  ┆ 3      ┆ 0           ┆ 997b3806bf ┆ … ┆ computer   ┆ Q: In      ┆ C          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ b3898e     ┆   ┆ science    ┆ building a ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ linear     ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ regres…    ┆            ┆            │
│ 250  ┆ 3      ┆ 41          ┆ 79519a07c0 ┆ … ┆ biology    ┆ Q: How     ┆ D          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 6089b1     ┆   ┆            ┆ does the   ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ term       ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ "growth" … ┆            ┆            │
│ 251  ┆ 3      ┆ 39          ┆ 94dab38a5c ┆ … ┆ philosophy ┆ Q: Which   ┆ E          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 60442d     ┆   ┆            ┆ of the     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ following  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ is N…      ┆            ┆            │
│ 252  ┆ 3      ┆ 18          ┆ e810274adb ┆ … ┆ health     ┆ Q: Under   ┆ F          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ a8434c     ┆   ┆            ┆ which circ ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ umstances  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ w…         ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 8485 ┆ 84     ┆ 90          ┆ b0ca6d280a ┆ … ┆ biology    ┆ Q: What    ┆ D          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 265627     ┆   ┆            ┆ are the    ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ bacterial  ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ fact…      ┆            ┆ …          │
│ 8486 ┆ 84     ┆ 98          ┆ 1ce5acbc14 ┆ … ┆ economics  ┆ Q: In a    ┆ A          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ db9998     ┆   ┆            ┆ given      ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ economy,   ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ househo…   ┆            ┆ …          │
│ 8487 ┆ 84     ┆ 99          ┆ 4f1f24d37b ┆ … ┆ physics    ┆ Q: Camera  ┆ H          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 56c4f7     ┆   ┆            ┆ lenses     ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ usually    ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ conta…     ┆            ┆ …          │
│ 8488 ┆ 84     ┆ 87          ┆ aa5e5ab577

## Unparsable answers:

In [ ]:
unparsable_answers_per_model = defaultdict(int)

### In the baseline:

In [ ]:
for answer in (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
                .filter(pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___"))
                .select(["final_answer", "model_name"])
                .iter_rows()
):
    print(answer[1], ": \n")
    print(answer[0][-150:])
    print("-" * 50)
    unparsable_answers_per_model[answer[1]] += 1

## In the MAD:

In [ ]:
for answer in chain(
        mad_analysis.filter(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value)
                    .list.eval(pl.element().str.starts_with("___"))
                    .list.any()
        )
                .select(
            "answers_at_beginning",
            AnalysisColumn.parsed_individual_answers_before.value,
            "model_names",
        )
                .iter_rows(),
        mad_analysis.filter(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value)
                    .list.eval(pl.element().str.starts_with("___"))
                    .list.any()
        )
                .select(
            "answers_at_end",
            AnalysisColumn.parsed_individual_answers_after.value,
            "model_names",
        )
                .iter_rows(),
):
    for i, parsed in enumerate(answer[1]):
        if parsed.startswith("___"):
            unparsable_answers_per_model[answer[2][i]] += 1
            print(parsed, f"from {answer[2][i]}: \n")
            print(answer[0][i][-150:])
            print("-" * 50)

-> Qwen 0.6B often says "Answer should be A /think. The Answer is G"

In [ ]:
unparsable_answers_per_model

### Build an overview of parsing error Influence

In [ ]:
baseline_frame_with_group_constellation = baseline_frame.with_columns(
    group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING)
                        + pl.lit(" (Baseline)")
).drop("model_name")

parsing_error_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for handling in ["null", "wrong", "random"]:
    new_comparer = AnswerComparer(AnswerOptions.letters_A_to_J, handling)

    new_mad = (
        apply_parsing_and_group_decision(mad_frame, parser, new_comparer, group_reply)
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    new_base = (
        baseline_frame_with_group_constellation.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            is_correct=new_comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            )
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    parsing_error_influence_table = parsing_error_influence_table.join(
        pl.concat([new_mad, new_base], how="diagonal").select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({handling})"),
        ),
        on="group_constellation",
        how="inner",
    )

parsing_error_influence_table.with_columns(
    deviation=(
            pl.max_horizontal(pl.exclude("group_constellation"))
            - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

-> Using (null) is the best, as unparsed values increase your score... should not be used

-> Using wrong is the worst, could possibly be used to be fair, as it is "not correct"

-> Papers and Benchmarks often use "random", which increases the values artificially and introduces noise.. I do not like it, but to be fair one should use it.

---

-> But in all of out cases, even absolute deviation is actually pretty low. (it gets mitigated in group decisions, as unparsable values are ignored in aggregation)

# Agreeableness -> Number of Cases where they reach conslusion

In [ ]:
unanimity = (
    mad_analysis.with_columns(
        pl.col(AnalysisColumn.parsed_individual_answers_after.value)
        .list.n_unique()
        .eq(1)
        .alias("End unanimity"),
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.n_unique()
        .eq(1)
        .alias("Start unanimity"),
    )
    .group_by("group_constellation")
    .agg(
        pl.col("End unanimity").mean(),
        pl.col("Start unanimity").mean(),
        pl.when(pl.col("Start unanimity"))
        .then(None)
        .otherwise(pl.col("End unanimity"))
        .mean()
        .alias("End Unanimity | not Start Unanimity"),
        pl.when(pl.col("Start unanimity"))
        .then(pl.col("End unanimity"))
        .otherwise(None)
        .mean()
        .alias("End Unanimity | Start Unanimity"),
        accuracy=pl.col("is_correct").mean(),
    )
)

unanimity

-> In Human Groups (see Group Problem Solving) there is the tendency that the smarter the group, the more it is a "Truth Supported" Decision Scheme, the "dumber" the group, the more it is "proportional"

### Correlation for groups (ignoring 1 member, as it is always 1)

In [ ]:
print("Pearson Correlation:")
print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .corr()
)

print("Spearman Rank Correlation:")

print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

-> significantly correlated

---
-> Accuracy Correlates with Unanimity
Of course this could be that if everyone is correct in the beginning, then if they that way, then the chance of being correct is higher,
But can we increase the accuracy by accepting when a group starts with one answer?

## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)

In [ ]:
correctness_influence = (
    mad_analysis.with_columns(
        correct_before=comparer(
            pl.col(AnalysisColumn.parsed_combined_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("group_constellation")
    .agg(
        pl.when(pl.col("correct_before"))
        .then(pl.col("is_correct"))
        .otherwise(None)
        .mean()
        .alias("Correct | Correct in Beginning"),
        pl.when(pl.col("correct_before"))
        .then(None)
        .otherwise(pl.col("is_correct"))
        .mean()
        .alias("Correct | !Correct in Beginning"),
        accuracy=pl.col("is_correct").mean(),
    )
)

correctness_influence

While the difference in Correct | Correct in Beginning is negligible, the true power lies in changing the answer when They are not correct.

In [ ]:
(
    correctness_influence.join(unanimity, how="left", on="group_constellation")
    .filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

## Influence of Group Aggregation Protocol

In [ ]:
group_reply_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for strategy in [MajorityVote(), SingularityVote()]:
    new_group_reply_agg = GroupReplyAggregator(strategy)
    group_reply_influence_table = group_reply_influence_table.join(
        apply_parsing_and_group_decision(
            mad_frame, parser, comparer, new_group_reply_agg
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
        .select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({strategy.__class__.__name__})"),
        ),
        on="group_constellation",
        how="inner",
    )

group_reply_influence_table = group_reply_influence_table.with_columns(
    deviation=(
            pl.max_horizontal(pl.exclude("group_constellation"))
            - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_reply_influence_table

In [27]:
group_reply_influence_table.drop("group_constellation").with_columns(
    pl.all().rank()
).corr()

Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy_right
f64,f64,f64,f64,f64,f64,f64,f64
1.0,0.01958,0.714517,0.472904,0.351131,0.463742,0.419853,0.714517
0.01958,1.0,-0.12255,-0.351177,-0.391212,-0.245079,-0.149016,-0.12255
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0
0.472904,-0.351177,0.493686,1.0,0.8586,0.878381,0.618211,0.493686
0.351131,-0.391212,0.349566,0.8586,1.0,0.754214,0.455629,0.349566
0.463742,-0.245079,0.502777,0.878381,0.754214,1.0,0.297922,0.502777
0.419853,-0.149016,0.285166,0.618211,0.455629,0.297922,1.0,0.285166
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0


-> Slightly Negative Correlation between Accuracy and the deviation, meaning the better the more MajorityVote == SingularityVote -> Same argument as before

## Inter-Model Correctness Correlation (Aka answer diversity)

In [ ]:
mad_analysis

#### For Individual answers

In [ ]:
individual_models_answer_per_question = (
    (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
            is_correct=comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            ),
        )
        .pivot(
            values="is_correct",
            index="question_id",
            on="model_name",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .select("question_id", "L", "M", "H")
    .with_columns(pl.exclude("question_id").cast(bool))
)

individual_models_answer_per_question

In [30]:
individual_models_answer_per_question.drop("question_id").corr()

shape: (3_400, 23)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ group_cons ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ tellation  ┆ combined_a ┆ combined_a ┆ ---        │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ ---        ┆ nswers_bef ┆ nswers_aft ┆ bool       │
│      ┆        ┆             ┆ str        ┆   ┆ str        ┆ …          ┆ …          ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ ---        ┆ ---        ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ str        ┆ str        ┆            │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 294  ┆ 3      ┆ 45          ┆ ee9c12870a ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ 3b979e     ┆   ┆            ┆            ┆            ┆            │
│ 295  ┆ 3      ┆ 79          ┆ 1d6ae1faca ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ cf8002     ┆   ┆            ┆            ┆            ┆            │
│ 296  ┆ 3      ┆ 95          ┆ f9d891b522 ┆ … ┆ H          ┆ C          ┆ C          ┆ false      │
│      ┆        ┆             ┆ e3eec7     ┆   ┆            ┆            ┆            ┆            │
│ 297  ┆ 3      ┆ 22          ┆ 0bdce6238f ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ 1ca68f     ┆   ┆            ┆            ┆            ┆            │
│ 298  ┆ 3      ┆ 61          ┆ 607c013c34 ┆ … ┆ H          ┆ H          ┆ H          ┆ true       │
│      ┆        ┆             ┆ 516755     ┆   ┆            ┆            ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 3695 ┆ 36     ┆ 52          ┆ 8599cb5bf0 ┆ … ┆ LMM        ┆ D          ┆ D          ┆ true       │
│      ┆        ┆             ┆ c91c9d     ┆   ┆            ┆            ┆            ┆            │
│ 3696 ┆ 36     ┆ 5           ┆ b113d185f8 ┆ … ┆ LMM        ┆ C          ┆ C          ┆ false      │
│      ┆        ┆             ┆ 688d64     ┆   ┆            ┆            ┆            ┆            │
│ 3697 ┆ 36     ┆ 9           ┆ a82a93c4d7 ┆ … ┆ LMM        ┆ I          ┆ I          ┆ true       │
│      ┆        ┆             ┆ a8be41     ┆   ┆            ┆            ┆            ┆            │
│ 3698 ┆ 36     ┆ 83          ┆ 3f971bfed7 ┆ … ┆ LMM        ┆ C          ┆ C          ┆ true       │
│      ┆        ┆             ┆ 2f5e1c     ┆   ┆            ┆            ┆            ┆            │
│ 3699 ┆ 36     ┆ 90          ┆ e0ea217990 ┆ … ┆ LMM        ┆ D          ┆ D          ┆ true       │
│      ┆        ┆             ┆ 89f5d4     ┆   ┆            ┆            ┆            ┆            │
└──────┴────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

--> Actually surprisingly different

In [ ]:
df = pl.concat(
    [
        individual_models_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
)

df_single = df.select("Given", "L", "M", "H")

df_single

-> They are not completely overlapping in the single answer case.

This means could be some diversity effect going on

---

In [ ]:
individual_groups_answer_per_question = (
    (
        mad_analysis.filter(pl.col("group_constellation").str.len_chars() == 1).pivot(
            values="is_correct",
            index="question_id",
            on="group_constellation",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .with_columns(pl.exclude("question_id").cast(bool))
)

df_single_mad = pl.concat(
    [
        individual_groups_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", "L", "M", "H")

df_single_mad

In [ ]:
combined_ind = individual_models_answer_per_question.join(
    individual_groups_answer_per_question, on="question_id", suffix="_mad"
)
pl.concat(
    [
        combined_ind.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in combined_ind.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", pl.exclude("Given"))

> See Obsidian

#### Is the "nobody is right case" in HHH actually unanimous?

In [ ]:
print("Answers where everyone was wrong:")

everyone_wrong = (
    mad_analysis.filter(pl.col("group_constellation") == "HHH")
    .explode(AnalysisColumn.parsed_individual_answers_before.value)
    .with_columns(
        is_individually_correct=comparer(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("question_id")
    .agg(
        number_wrong=pl.len(),
        answers=pl.col(AnalysisColumn.parsed_individual_answers_before.value).implode(),
        is_individually_correct=pl.col("is_individually_correct").implode(),
    )
    .filter(pl.col("is_individually_correct").list.eval(pl.element().not_()).list.all())
    .drop("number_wrong", "is_individually_correct")
)

print(everyone_wrong)
unanimous = (
    everyone_wrong["answers"]
    .filter(everyone_wrong["answers"].list.n_unique() == 1)
    .count()
)
print(f"Of that unanimous: {unanimous}")
print("Contentious:")
print(everyone_wrong["answers"].filter(everyone_wrong["answers"].list.n_unique() != 1))

C# Intra-Group Correctness Correlation (Aka group diversity)

Answers where everyone was wrong:
shape: (23, 2)
┌─────────────┬─────────────────────────────────┐
│ question_id ┆ answers                         │
│ ---         ┆ ---                             │
│ i64         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ 4           ┆ ["D", "D", "D"]                 │
│ 92          ┆ ["C", "D", "C"]                 │
│ 43          ┆ ["___not_parsable___", "___not… │
│ 74          ┆ ["D", "E", "E"]                 │
│ 113         ┆ ["E", "A", "___not_parsable___… │
│ …           ┆ …                               │
│ 6           ┆ ["D", "D", "D"]                 │
│ 13          ┆ ["E", "E", "E"]                 │
│ 109         ┆ ["H", "H", "H"]                 │
│ 3           ┆ ["C", "C", "C"]                 │
│ 30          ┆ ["A", "A", "A"]                 │
└─────────────┴─────────────────────────────────┘
Of that unanimous: 15
Contentious:
shape: (8,)
Series: 'answers' [list[str]]
[
	["C", "D", "C"]
	["D"

C# Intra-Group Correctness Correlation (Aka group diversity)